[//]: # (cr:doc name='parity_audit_intro' id=35807d48)
# Parity Audit (T0 Pre-flight)

This notebook is the first task of every Databricks job. It scans the
exploration notebooks `NB00`-`NB09` for `@apply_op`-decorated call sites,
renders the production pipeline in-memory, and reconciles the two
manifests. If a divergence is detected (e.g. production would apply a
lookback that exploration skipped), the job aborts before any data
moves.

Spec: `docs/wiki/Parity-Audit.md`.

In [ ]:
# @cr:code name='parity_audit_setup' id=63f6c587
from __future__ import annotations

import os
import sys
from pathlib import Path

FRAMEWORK_REPO_ROOT = os.environ.get("FRAMEWORK_REPO_ROOT")
if FRAMEWORK_REPO_ROOT:
    sys.path.insert(0, f"{FRAMEWORK_REPO_ROOT}/src")

from customer_retention.core.config.experiments import get_experiments_dir
from customer_retention.parity import (
    AuditScope,
    audit_landing,
)

ENGAGEMENT_DIR = Path(os.environ.get("CR_ENGAGEMENT_DIR", "."))
PIPELINE_DIR = Path(
    os.environ.get("CR_PIPELINE_DIR")
    or (get_experiments_dir() / "generated_pipeline")
)

print(f"engagement: {ENGAGEMENT_DIR.resolve()}")
print(f"pipeline:   {PIPELINE_DIR.resolve()}")

In [ ]:
# @cr:code name='parity_audit_run' id=fe210ab1
outcome = audit_landing(
    engagement_dir=ENGAGEMENT_DIR,
    pipeline_dir=PIPELINE_DIR,
)
print(outcome.format_report())

In [ ]:
# @cr:code name='parity_audit_exit' id=9853af36
if outcome.has_gaps:
    try:
        dbutils.notebook.exit(outcome.to_failed_json())  # noqa: F821
    except NameError:
        raise SystemExit(outcome.exit_code)
else:
    print(outcome.format_summary())